# RAG From Scratch: Long Context vs RAG

Long-context models changed the RAG tradeoff. A practical system should route between direct long-context answering and retrieval based on corpus size, latency, cost, and query type.

In [ ]:
import sys
!uv pip install --python {sys.executable} -r pyproject.toml

## Model provider

Shared provider selector for notebook generation cells. `RAG_CHAT_PROVIDER=gemini` keeps the original Gemini chat path. Set `RAG_CHAT_PROVIDER=openrouter` and `OPENROUTER_API_KEY` in `.env` to use OpenRouter cheap models. Retrieval examples default to `RAG_EMBEDDING_PROVIDER=local` for API-free TF-IDF embeddings; set `RAG_EMBEDDING_PROVIDER=gemini` for the original Gemini embeddings.


In [ ]:
from rag_providers import (
    OPENROUTER_MODELS,
    chat_model,
    chat_provider_name,
    embedding_model,
    embedding_provider_name,
    openrouter_model_name,
)

print("Chat provider:", chat_provider_name())
print("Embedding provider:", embedding_provider_name())
print("OpenRouter model:", openrouter_model_name())
OPENROUTER_MODELS


In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(encoding.encode(text))

corpus = [
    "RAG is cheaper when only a few chunks are needed.",
    "Long context can win when the answer needs many dispersed facts.",
    "Hybrid routing asks whether retrieval is sufficient before paying for full-context reads.",
]

corpus_text = "\n\n".join(corpus)
count_tokens(corpus_text)

In [ ]:
def route_query(question, corpus_text, long_context_budget=3000):
    q = question.lower()
    corpus_tokens = count_tokens(corpus_text)
    needs_global_read = any(term in q for term in ["summarize", "compare all", "across the corpus", "overall"])
    if corpus_tokens <= long_context_budget and needs_global_read:
        return "long_context"
    return "rag"

questions = [
    "What is the cheapest way to answer a narrow fact question?",
    "Summarize the overall tradeoff across the corpus.",
]

{question: route_query(question, corpus_text) for question in questions}

In [ ]:
def pack_long_context(question, corpus):
    return f"Answer using the full corpus.\n\nCorpus:\n{chr(10).join(corpus)}\n\nQuestion: {question}"

def pack_rag_context(question, retrieved_chunks):
    return f"Answer using only the retrieved context.\n\nContext:\n{chr(10).join(retrieved_chunks)}\n\nQuestion: {question}"

question = questions[1]
mode = route_query(question, corpus_text)
prompt = pack_long_context(question, corpus) if mode == "long_context" else pack_rag_context(question, corpus[:1])
print(mode)
print(prompt)

Production systems normally log the route decision, prompt tokens, output tokens, latency, and eval score. That creates the data needed to tune the router instead of guessing.